# Model 3 V2 — Group-Aware Evaluation

Evaluates `network_threat_model_v2_group_aware.pkl` against the four completely held-out source files.

Reports:
- binary precision/recall/F1
- FPR and benign detection rate for benign-only files
- family macro F1 for families the training set actually knows
- separate reporting of unseen-family rows
- per-file results
- aggregate metrics

No production artifact is modified.

In [4]:
from pathlib import Path
import json, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_DIR = Path(r"C:\Users\bimal\OneDrive\Desktop\EVIDENT")

# Actual CIC-IDS2017 dataset location
RAW_DIR = PROJECT_DIR / "data" / "raw" / "behaviour"

MODEL_PATH = (
    PROJECT_DIR
    / "models"
    / "behavior"
    / "experiments"
    / "network_threat_model_v2_group_aware.pkl"
)

META_DIR = PROJECT_DIR / "models" / "behavior" / "metadata"

print("Dataset directory:", RAW_DIR)
print("Dataset exists:", RAW_DIR.exists())
print("Model:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

if not RAW_DIR.exists():
    raise FileNotFoundError(
        f"CIC-IDS2017 dataset directory not found:\n{RAW_DIR}"
    )

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model artifact not found:\n{MODEL_PATH}"
    )

bundle = joblib.load(MODEL_PATH)

print("\nLoaded:", MODEL_PATH)
print("Training groups:", bundle["training_groups"])
print("Known family labels:", bundle["family_labels"])

print("\nAvailable CSV files:")
for p in sorted(RAW_DIR.glob("*.csv")):
    print(" -", p.name)

Dataset directory: C:\Users\bimal\OneDrive\Desktop\EVIDENT\data\raw\behaviour
Dataset exists: True
Model: C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_v2_group_aware.pkl
Model exists: True

Loaded: C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\experiments\network_threat_model_v2_group_aware.pkl
Training groups: ['Monday-WorkingHours.pcap_ISCX.csv', 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'Tuesday-WorkingHours.pcap_ISCX.csv', 'Wednesday-workingHours.pcap_ISCX.csv']
Known family labels: ['BRUTE_FORCE', 'DOS_DDOS', 'HEARTBLEED', 'WEB_ATTACK']

Available CSV files:
 - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
 - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
 - Friday-WorkingHours-Morning.pcap_ISCX.csv
 - Monday-WorkingHours.pcap_ISCX.csv
 - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
 - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
 - Tuesday-WorkingHours.pcap_ISCX.csv
 - Wednesda

In [5]:
def label_col(df):
    return next(c for c in df.columns if str(c).strip().lower()=="label")
FAMILY_MAP={
"BENIGN":"BENIGN","DDoS":"DOS_DDOS","DoS Hulk":"DOS_DDOS","DoS GoldenEye":"DOS_DDOS",
"DoS slowloris":"DOS_DDOS","DoS Slowhttptest":"DOS_DDOS","PortScan":"PORT_SCAN",
"FTP-Patator":"BRUTE_FORCE","SSH-Patator":"BRUTE_FORCE",
"Web Attack – Brute Force":"WEB_ATTACK","Web Attack – XSS":"WEB_ATTACK","Web Attack – Sql Injection":"WEB_ATTACK",
"Bot":"BOTNET","Infiltration":"INFILTRATION","Heartbleed":"HEARTBLEED"
}
def family(x):
    s=str(x).strip()
    if s in FAMILY_MAP:return FAMILY_MAP[s]
    sl=s.lower()
    if "web attack" in sl:return "WEB_ATTACK"
    if "patator" in sl:return "BRUTE_FORCE"
    if "dos" in sl or "ddos" in sl:return "DOS_DDOS"
    if "portscan" in sl:return "PORT_SCAN"
    if "infiltration" in sl:return "INFILTRATION"
    if "heartbleed" in sl:return "HEARTBLEED"
    if "bot" in sl:return "BOTNET"
    if sl=="benign":return "BENIGN"
    return "OTHER_ATTACK"

def predict(df):
    X=df.reindex(columns=bundle["feature_columns"]).replace([np.inf,-np.inf],np.nan)
    bm=bundle["binary_model"]
    p=bm.predict_proba(X)[:,1]
    pred=(p>=bundle["binary_threshold"]).astype(int)
    fpred=np.array(["BENIGN"]*len(df),dtype=object)
    mask=pred==1
    if mask.any():
        fpred[mask]=bundle["family_model"].predict(X.loc[mask])
    return pred,p,fpred

In [6]:
HOLDOUT=bundle["holdout_groups"]
results=[]
all_known_y=[]; all_known_p=[]; all_known_ftrue=[]; all_known_fpred=[]
for name in HOLDOUT:
    path=RAW_DIR/name
    df=pd.read_csv(path,low_memory=False)
    lc=label_col(df)
    yb=(df[lc].astype(str).str.strip().str.upper()!="BENIGN").astype(int).to_numpy()
    yf=df[lc].map(family).to_numpy()
    pred,p,fpred=predict(df)
    binary_f1=float(f1_score(yb,pred,zero_division=0)) if yb.sum() and pred.sum() else 0.0
    benign_mask=yb==0
    fpr=float((pred[benign_mask]==1).mean()) if benign_mask.any() else None
    known_mask=(yb==1) & np.isin(yf,bundle["family_labels"])
    fam_f1=float(f1_score(yf[known_mask],fpred[known_mask],average="macro",zero_division=0)) if known_mask.any() else None
    unseen_families=sorted(set(yf[(yb==1)])-set(bundle["family_labels"]))
    results.append({
        "file":name,"rows":len(df),"binary_f1":binary_f1,
        "false_positive_rate":fpr,"known_family_macro_f1":fam_f1,
        "attack_families":sorted(set(yf[yb==1])),
        "unseen_attack_families":unseen_families
    })
    print(results[-1])
    all_known_y.extend(yb[known_mask].tolist()); all_known_p.extend(pred[known_mask].tolist())
    all_known_ftrue.extend(yf[known_mask].tolist()); all_known_fpred.extend(fpred[known_mask].tolist())

res=pd.DataFrame(results)
display(res)

{'file': 'Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv', 'rows': 288602, 'binary_f1': 0.0, 'false_positive_rate': 0.0010361581059445673, 'known_family_macro_f1': None, 'attack_families': ['INFILTRATION'], 'unseen_attack_families': ['INFILTRATION']}
{'file': 'Friday-WorkingHours-Morning.pcap_ISCX.csv', 'rows': 191033, 'binary_f1': 0.0, 'false_positive_rate': 0.000782791285628904, 'known_family_macro_f1': None, 'attack_families': ['BOTNET'], 'unseen_attack_families': ['BOTNET']}
{'file': 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv', 'rows': 286467, 'binary_f1': 0.006655010997687665, 'false_positive_rate': 0.0009252217003693046, 'known_family_macro_f1': None, 'attack_families': ['PORT_SCAN'], 'unseen_attack_families': ['PORT_SCAN']}
{'file': 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv', 'rows': 225745, 'binary_f1': 0.3275618928235663, 'false_positive_rate': 0.0005628441024171596, 'known_family_macro_f1': 0.16383977846427147, 'attack_families': ['DOS_DDOS'], 

,file,rows,binary_f1,false_positive_rate,known_family_macro_f1,attack_families,unseen_attack_families
0,Thursday-WorkingHours-Afternoon-Infilteration....,288602,0.000000,0.001036,NaN,[INFILTRATION],[INFILTRATION]
1,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,0.000000,0.000783,NaN,[BOTNET],[BOTNET]
2,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,0.006655,0.000925,NaN,[PORT_SCAN],[PORT_SCAN]
3,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,0.327562,0.000563,0.16384,[DOS_DDOS],[]


In [7]:
aggregate={
"average_binary_f1":float(res["binary_f1"].mean()),
"minimum_binary_f1":float(res["binary_f1"].min()),
"average_known_family_macro_f1":float(res["known_family_macro_f1"].dropna().mean()) if res["known_family_macro_f1"].notna().any() else None,
"minimum_known_family_macro_f1":float(res["known_family_macro_f1"].dropna().min()) if res["known_family_macro_f1"].notna().any() else None,
"mean_false_positive_rate":float(res["false_positive_rate"].dropna().mean()) if res["false_positive_rate"].notna().any() else None,
"known_family_labels":bundle["family_labels"],
"holdout_groups":HOLDOUT
}
out={"aggregate":aggregate,"per_file":res.to_dict(orient="records")}
path=META_DIR/"model3_v2_group_aware_validation.json"
path.write_text(json.dumps(out,indent=2),encoding="utf-8")
print("=== MODEL 3 V2 GROUP-AWARE QUALITY GATE ===")
print(json.dumps(aggregate,indent=2))
print("Saved:",path)

=== MODEL 3 V2 GROUP-AWARE QUALITY GATE ===
{
  "average_binary_f1": 0.08355422595531349,
  "minimum_binary_f1": 0.0,
  "average_known_family_macro_f1": 0.16383977846427147,
  "minimum_known_family_macro_f1": 0.16383977846427147,
  "mean_false_positive_rate": 0.0008267537985899839,
  "known_family_labels": [
    "BRUTE_FORCE",
    "DOS_DDOS",
    "HEARTBLEED",
    "WEB_ATTACK"
  ],
  "holdout_groups": [
    "Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
  ]
}
Saved: C:\Users\bimal\OneDrive\Desktop\EVIDENT\models\behavior\metadata\model3_v2_group_aware_validation.json
